# 09 Frozen Canonical Actual Path

This notebook formalises the academic firewall for the synthetic 15-minute realized market path used in downstream bidding work.

Current scope:

- build and freeze one canonical synthetic 15-minute actual path;
- keep that path separate from forecast and optimisation artifacts;
- audit timestamp completeness, hourly reconciliation, randomness governance, and input independence;
- expose the manifest, diagnostics, and checksum that later notebooks must report.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## Why This Path Exists

A full historical 15-minute DA year is not available for the official hourly test period. The repository therefore uses a single frozen counterfactual 15-minute market environment for downstream bidding experiments.

This is not a historical validation object. It is a pre-specified, reproducible, and versioned realized-path assumption.

## Optional Frozen-Actual Runner

In [ ]:
RUN_FROZEN_ACTUAL = False

if RUN_FROZEN_ACTUAL:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / "run_15min_frozen_actual_path.py"),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Frozen canonical actual-path build failed with exit code {completed.returncode}.")

In [ ]:
VERSION_ID = "canonical_v1"

version_dir = find_frozen_actual_version(config, VERSION_ID)
if version_dir is None:
    raise FileNotFoundError(f"No frozen canonical actual version '{VERSION_ID}' exists yet. Run the frozen-actual script first.")

registry_entry = resolve_frozen_actual_registry_entry(config, version_id=VERSION_ID, verify_hash=True)
manifest = load_frozen_actual_manifest(config, version_id=VERSION_ID, verify_hash=False)
diagnostics = load_frozen_actual_diagnostics(config, version_id=VERSION_ID, verify_hash=False)
canonical_actual = load_frozen_actual_path(config, version_id=VERSION_ID, verify_hash=False)
provenance = build_thesis_grade_frozen_actual_metadata(config, version_id=VERSION_ID, verify_hash=False)

display(
    pd.DataFrame(
        [
            {
                "version_id": VERSION_ID,
                "version_dir": str(version_dir),
                "manifest_path": str(registry_entry.manifest_path),
                "csv_sha256": registry_entry.authoritative_sha256,
            }
        ]
    )
)

## Thesis-Grade Provenance Payload

In [ ]:
display(pd.DataFrame([provenance]))

## Manifest

In [ ]:
manifest_rows = []
for key, value in manifest.items():
    if isinstance(value, (dict, list)):
        manifest_rows.append({"field": key, "value": json.dumps(value, default=str)})
    else:
        manifest_rows.append({"field": key, "value": value})
display(pd.DataFrame(manifest_rows))

## Audit Diagnostics

In [ ]:
display(diagnostics)

## Canonical Path Preview

In [ ]:
display(canonical_actual.head(12))

## Interpretation Contract

What the implemented checks prove:

- the canonical path is complete in UTC quarter-hour timestamps;
- the synthetic quarter-hour path reconciles back to the observed hourly DA anchor;
- the canonical path records a fixed seed, explicit version id, and checksum;
- the canonical path is documented as independent from forecast outputs, optimisation outputs, and economic selection.

What the checks do not prove:

- that the synthetic path is the true realized 15-minute historical market for the official hourly test year;
- that one canonical path fully spans all plausible market environments.

## Required Downstream Usage

- Forecast/scenario notebooks write separate ex-ante artifacts.
- Bidding notebooks load this frozen canonical actual path as read-only input.
- Evaluation notebooks report the version id and manifest hash used.